In [2]:
import os

# 1. Define the path to our good Java
java21_path = "/usr/lib/jvm/java-21-openjdk-amd64"

# 2. Set JAVA_HOME
os.environ["JAVA_HOME"] = java21_path

# 3. The Magic: Shove Java 21 to the absolute front of the PATH
os.environ["PATH"] = f"{java21_path}/bin:" + os.environ.get("PATH", "")

In [3]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .config("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh") \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/12 20:30:03 WARN Utils: Your hostname, codespaces-472bba, resolves to a loopback address: 127.0.0.1; using 10.0.10.7 instead (on interface eth0)
26/03/12 20:30:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 20:30:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df_green = spark.read \
    .option ("recursiveFileLookup", "True") \
    .parquet('data/pq/green')

In [6]:
# Instead of deprecated RegisterTempTable, we create a TempView
df_green.createOrReplaceTempView('green')

```
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
```

In [7]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [24]:
from datetime import datetime

start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value) 

def calculate_revenue(left_v, right_v):
    left_amount, left_count = left_v
    right_amount, right_count = right_v

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)

from collections import namedtuple

RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

def unwrap(row):
    return RevenueRow(
        hour = row[0][0],
        zone = row[0][1],
        revenue = row[1][0],
        count = row[1][1]
        )

In [29]:
from pyspark.sql import types

results_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True), 
    types.StructField('zone', types.IntegerType(), True), 
    types.StructField('revenue', types.DoubleType(), True), 
    types.StructField('count', types.IntegerType(), True)
])

In [32]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(results_schema)

In [31]:
df_result.schema

StructType([StructField('hour', TimestampType(), True), StructField('zone', IntegerType(), True), StructField('revenue', DoubleType(), True), StructField('count', IntegerType(), True)])

In [33]:
df_result.show()

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-23 20:00:00|  74|1044.0499999999993|   60|
|2020-01-15 18:00:00| 179|              50.5|    5|
|2020-01-16 15:00:00|  41| 736.1399999999996|   54|
|2020-01-31 18:00:00| 260|            126.32|    7|
|2020-01-02 15:00:00|  66|            197.69|   10|
|2020-01-16 21:00:00|  74| 895.1799999999994|   60|
|2020-01-23 19:00:00|  37|             43.35|    2|
|2020-01-11 05:00:00|  95| 407.7100000000002|   37|
|2020-01-14 02:00:00| 223|213.83000000000004|   19|
|2020-01-22 01:00:00| 166| 685.3899999999999|   50|
|2020-01-23 22:00:00| 166| 901.6799999999995|   59|
|2020-01-08 01:00:00|  25| 554.2900000000001|   37|
|2020-01-01 08:00:00| 181| 730.1499999999997|   25|
|2020-01-18 14:00:00|  55|              48.3|    1|
|2020-01-20 07:00:00|  74|221.33000000000004|   16|
|2020-01-11 23:00:00|  65|            936.16|   43|
|2020-01-03 

Traceback (most recent call last):
  File "/workspaces/de-zoomcamp-26/06-batch/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/workspaces/de-zoomcamp-26/06-batch/.venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [34]:
df_result.write.parquet('tmp/revenue-green')

In [ ]:
# Map Partition

In [35]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [41]:
import pandas as pd

In [53]:
rows = duration_rdd.take(10)
rows

pd.DataFrame(rows, columns = columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2.0,2020-01-23 13:10:15,74,130,12.77
1,NaN,2020-01-20 15:09:00,67,39,8.00
2,2.0,2020-01-15 20:23:41,260,157,1.27
3,2.0,2020-01-05 16:32:26,82,83,1.25
4,2.0,2020-01-29 19:22:42,166,42,1.84
5,2.0,2020-01-15 11:07:42,179,223,0.76
6,2.0,2020-01-16 08:22:29,41,237,3.32
7,2.0,2020-01-28 17:05:28,75,161,2.21
8,1.0,2020-01-22 14:51:37,152,166,0.90
9,2.0,2020-01-31 10:25:04,75,234,6.10


In [48]:
#model = ...

def model_predict(df):
#     y_pred = model.predict(df)
    y_pred = df.trip_distance * 5
    return y_pred

In [51]:
def apply_model_in_batch(partition):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions
    for row in df.itertuples():
        yield row

In [54]:
df_predictions = duration_rdd.mapPartitions(apply_model_in_batch).toDF()

In [57]:
df_predictions.drop('Index').show()
df_predictions.select('predicted_duration').show()

+--------+--------------------+------------+------------+-------------+------------------+
|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+--------+--------------------+------------+------------+-------------+------------------+
|     2.0|                  {}|          74|         130|        12.77|63.849999999999994|
|     NaN|                  {}|          67|          39|          8.0|              40.0|
|     2.0|                  {}|         260|         157|         1.27|              6.35|
|     2.0|                  {}|          82|          83|         1.25|              6.25|
|     2.0|                  {}|         166|          42|         1.84| 9.200000000000001|
|     2.0|                  {}|         179|         223|         0.76|               3.8|
|     2.0|                  {}|          41|         237|         3.32|16.599999999999998|
|     2.0|                  {}|          75|         161|         2.21|             11.05|

+------------------+
|predicted_duration|
+------------------+
|63.849999999999994|
|              40.0|
|              6.35|
|              6.25|
| 9.200000000000001|
|               3.8|
|16.599999999999998|
|             11.05|
|               4.5|
|              30.5|
|63.849999999999994|
|              40.0|
|              6.35|
|              6.25|
| 9.200000000000001|
|               3.8|
|16.599999999999998|
|             11.05|
|               4.5|
|              30.5|
+------------------+
only showing top 20 rows
